# 04 — Model Training and Hyper-parameter Tuning

> **This notebook loads the results of the hyper-parameter search rather than
> re-running it.** The search takes roughly an hour on 16 cores, and a notebook
> that takes an hour to run is a notebook nobody runs. The exact command that
> produced these artefacts is shown below, and re-running it reproduces them
> because every random state is fixed.

```bash
python -m src.train              # the main experiment
python -m src.train --ablation no_ttl
```

In [1]:
import sys, warnings
from pathlib import Path

# Make the repository root importable no matter where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

from src import config
print(f"Repository root : {ROOT}")
print(f"Random seed     : {config.RANDOM_STATE}")

Repository root : E:\AIM\AIM AI ML Capstone\AI_Capstone_Network_Intrusion_Detection
Random seed     : 42


## 1. The models and why each is here

In [2]:
from src import train as train_module

registry = train_module.model_registry()
for key, spec in registry.items():
    print(f"=== {spec.display_name} ===")
    print(f"  tags       : {', '.join(spec.tags)}")
    print(f"  candidates : {spec.n_iter} x {config.CV_FOLDS} folds")
    print(f"  rationale  : {spec.notes}")
    print()

=== Logistic Regression ===
  tags       : linear, interpretable, baseline
  candidates : 16 x 5 folds
  rationale  : Interpretable linear baseline. Coefficients are directly readable as log-odds contributions, which makes it the reference point for judging whether the ensembles' extra complexity is earning its keep.

=== Random Forest ===
  tags       : ensemble, bagging, tree
  candidates : 25 x 5 folds
  rationale  : Bagged axis-aligned trees. Robust to the dataset's heavy tails and mixed feature scales, and provides variance reduction without the sequential-fitting cost of boosting.

=== XGBoost ===
  tags       : ensemble, boosting, tree
  candidates : 30 x 5 folds
  rationale  : Gradient-boosted trees. Expected to lead on tabular flow data; the primary candidate for deployment and the model carried into the SHAP analysis.

=== LightGBM ===
  tags       : ensemble, boosting, tree, optional
  candidates : 25 x 5 folds
  rationale  : Optional fourth model: leaf-wise boosting. Includ

In [3]:
print("Search spaces:\n")
for key, spec in registry.items():
    print(f"{spec.display_name}")
    for param, distribution in spec.param_distributions.items():
        name = param.removeprefix("model__")
        kind = (f"{type(distribution).__name__}" if hasattr(distribution, "rvs")
                else str(distribution))
        print(f"    {name:<22} {kind}")
    print()

Search spaces:

Logistic Regression
    C                      rv_continuous_frozen
    class_weight           [None, 'balanced']

Random Forest
    n_estimators           rv_discrete_frozen
    max_depth              [None, 12, 18, 24, 32]
    min_samples_leaf       rv_discrete_frozen
    min_samples_split      rv_discrete_frozen
    max_features           ['sqrt', 'log2', 0.3]
    class_weight           [None, 'balanced', 'balanced_subsample']

XGBoost
    n_estimators           rv_discrete_frozen
    max_depth              rv_discrete_frozen
    learning_rate          rv_continuous_frozen
    subsample              rv_continuous_frozen
    colsample_bytree       rv_continuous_frozen
    min_child_weight       rv_discrete_frozen
    reg_lambda             rv_continuous_frozen
    reg_alpha              rv_continuous_frozen
    gamma                  rv_continuous_frozen

LightGBM
    n_estimators           rv_discrete_frozen
    num_leaves             rv_discrete_frozen
    learning_

### Isolation Forest — the unsupervised comparison

Fitted on **benign training flows only**, with the attack labels withheld
entirely. That is the honest framing of anomaly detection for intrusion
detection: show the detector what normal looks like and make it flag everything
else.

It answers a question the supervised models cannot — *how far could we get with
no attack labels at all?* — and sets the floor that supervised learning has to
beat to justify its labelling cost.

## 2. Tuning protocol

- `RandomizedSearchCV` with 5-fold `StratifiedKFold`, `random_state=42`
- Scoring on **F1**, with average precision and ROC-AUC recorded alongside
- Each candidate is a **full `Pipeline`** — feature engineering → preprocessing →
  estimator

That last point is what makes fold-level leakage impossible. Every fold re-fits
the scaler and the one-hot vocabulary on its own training portion. A scaler
fitted once before cross-validation would leak fold information into the
validation folds.

In [4]:
import json

params_path = config.MODELS_DIR / "best_params_main.json"
if not params_path.exists():
    display(Markdown(
        "> ⚠️ No tuning results found. Run `python -m src.train` first."))
    results = {}
else:
    results = json.loads(params_path.read_text(encoding="utf-8"))
    rows = []
    for key, record in results.items():
        rows.append({
            "model": record.get("display_name", key),
            "cv_f1_mean": record.get("cv_f1_mean"),
            "cv_f1_std": record.get("cv_f1_std"),
            "cv_pr_auc": record.get("cv_average_precision_mean"),
            "cv_roc_auc": record.get("cv_roc_auc_mean"),
            "train_minus_test_f1": record.get("cv_train_minus_test_f1"),
            "search_s": record.get("search_seconds"),
            "fit_s": record.get("fit_seconds"),
        })
    display(pd.DataFrame(rows).set_index("model"))

,cv_f1_mean,cv_f1_std,cv_pr_auc,cv_roc_auc,train_minus_test_f1,search_s,fit_s
model,,,,,,,
Logistic Regression,0.904544,0.002683,0.960514,0.967638,0.00029,89.27,6.852
Random Forest,0.930242,0.002334,0.983801,0.985212,0.03964,1441.47,16.620
XGBoost,0.932829,0.002205,0.984747,0.986158,0.04415,726.82,4.683
LightGBM,0.933308,0.001542,0.985202,0.986561,0.06541,768.18,8.017
Isolation Forest (unsupervised),NaN,NaN,NaN,NaN,NaN,NaN,0.951


The `train_minus_test_f1` column is the gap between a model's score on data it
was fitted on and its cross-validated score. A large positive gap indicates
memorisation. Tree ensembles are expected to show one; what matters is that the
cross-validated score and the held-out test score agree — which notebook 05
confirms.

In [5]:
for key, record in results.items():
    if "best_params" in record:
        print(f"{record.get('display_name', key)}:")
        for param, value in record["best_params"].items():
            formatted = f"{value:.6g}" if isinstance(value, float) else value
            print(f"    {param:<22} {formatted}")
        print()

Logistic Regression:
    C                      73.9227
    class_weight           None

Random Forest:
    class_weight           balanced_subsample
    max_depth              18
    max_features           0.3
    min_samples_leaf       1
    min_samples_split      9
    n_estimators           362

XGBoost:
    colsample_bytree       0.926505
    gamma                  0.147224
    learning_rate          0.0393203
    max_depth              9
    min_child_weight       1
    n_estimators           455
    reg_alpha              0.0190685
    reg_lambda             0.169922
    scale_pos_weight       1.05262
    subsample              0.974462

LightGBM:
    colsample_bytree       0.746898
    learning_rate          0.0641409
    min_child_samples      19
    n_estimators           753
    num_leaves             143
    reg_lambda             0.0227066
    scale_pos_weight       1
    subsample              0.958305
    subsample_freq         0

Isolation Forest (unsupervised):
    n_e

## 3. Solver selection for Logistic Regression — decided by measurement

The default solver was not assumed. Four combinations were benchmarked on the
92,210 × 73 training matrix:

| Solver | Penalty | Time | Validation F1 | Converged |
|---|---|---|---|---|
| `lbfgs` | L2 | 3.6 s | 0.8566 | yes |
| `liblinear` | L2 | 10.0 s | 0.8560 | yes |
| `saga` | L2 | 101.8 s | 0.8561 | yes |
| `saga` | L1 | 199.4 s | 0.8562 | **no** |

L1 was dropped: 55× the runtime for a 0.0004 F1 difference, and it hit the
iteration cap. The cause is instructive — the TTL features make the classes very
nearly linearly separable, so the unregularised optimum runs off toward infinite
coefficients and the solver never settles.

## 4. Persisted artefacts

In [6]:
for path in sorted(config.MODELS_DIR.glob("*")):
    size = path.stat().st_size
    unit = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
    print(f"  {path.name:<42} {unit:>10}")

  .gitkeep                                       0.0 KB
  best_params_keep_duplicates.json               4.4 KB
  best_params_main.json                          5.5 KB
  best_params_no_engineered.json                 4.4 KB
  best_params_no_ttl.json                        4.4 KB
  best_params_official_split.json                4.4 KB
  best_params_official_split_raw.json            4.4 KB
  best_params_pooled_random.json                 4.4 KB
  best_params_smote.json                         4.4 KB
  deployment.json                                0.4 KB
  feature_reference.json                        21.5 KB
  isolation_forest.joblib                      948.1 KB
  isolation_forest_keep_duplicates.joblib      994.3 KB
  isolation_forest_no_engineered.joblib        886.7 KB
  isolation_forest_no_ttl.joblib               937.3 KB
  isolation_forest_official_split.joblib       997.4 KB
  isolation_forest_official_split_raw.joblib   997.4 KB
  isolation_forest_pooled_random.joblib        9

In [7]:
manifest_path = config.MODELS_DIR / "manifest_main.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for key, value in manifest.items():
        print(f"  {key:<28} {value}")

  experiment                   main
  description                  Primary protocol: the authors' published partition, each side deduplicated and train/test overlap removed.
  settings                     {'include_engineered': True, 'exclude': [], 'use_smote': False}
  random_state                 42
  cv_folds                     5
  tuning_scorer                f1
  n_train_rows                 80832
  n_val_rows                   20208
  train_attack_rate            0.48718
  val_attack_rate              0.48718
  scale_pos_weight_used        1.0526
  models                       ['isolation_forest', 'lightgbm', 'logistic_regression', 'random_forest', 'xgboost']


Each `.joblib` is a **complete `Pipeline`** — feature engineering, preprocessing
and estimator travel together. A caller passes raw UNSW-NB15-shaped records and
never has to reproduce any transformation, which removes the most common cause
of training/serving skew.

## 5. Validation performance (test split still untouched)

In [8]:
validation_path = config.METRICS_DIR / "validation_metrics_main.csv"
if validation_path.exists():
    validation = pd.read_csv(validation_path, index_col=0)
    columns = ["display_name", "recall", "precision", "f1", "roc_auc",
               "pr_auc", "false_positive_rate", "false_negative_rate"]
    display(validation[[c for c in columns if c in validation.columns]]
            .style.format({c: "{:.4f}" for c in columns[1:]}))
else:
    display(Markdown("> Run `python -m src.pipeline --evaluate` to populate this."))

,display_name,recall,precision,f1,roc_auc,pr_auc,false_positive_rate,false_negative_rate
model,,,,,,,,
logistic_regression,Logistic Regression,0.9689,0.8451,0.9028,0.9676,0.9607,0.1688,0.0311
random_forest,Random Forest,0.9583,0.9109,0.9340,0.9865,0.9849,0.0891,0.0417
xgboost,XGBoost,0.9542,0.9178,0.9357,0.9871,0.9855,0.0812,0.0458
lightgbm,LightGBM,0.9510,0.9222,0.9364,0.9874,0.9858,0.0762,0.0490
isolation_forest,Isolation Forest,0.3108,0.7580,0.4409,0.7486,0.6944,0.0943,0.6892


**These are validation numbers.** They are what model selection and threshold
tuning are allowed to use. The test split is opened once, in notebook 05, after
both choices are frozen.

**Next:** [`05_model_evaluation.ipynb`](05_model_evaluation.ipynb)